## Your pipeline, part 5: many-model ensemble (three larger YOLO models on two GPUs + 6-model fusion)
HW2, notebook 11 (Kaggle **GPU T4 x2**, ~4.5-5 h; at most ~6.3 h, a time guard skips a model rather than run over).
Goal: leaderboard 0.30+ (now 0.2869 with notebook 10's 3-model fusion, val 0.2469).

**Why.** Fusion has been the most reliable gain (09: 2 models, 10: 3 models), and the largest single-model gain came from
model size (YOLO11s -> YOLO11m: val 0.207 -> 0.226). So: more members, each stronger than the current ones and different
from them in size, resolution or training randomness:

| lane (GPU) | model | why | est. h (train + passes) |
|---|---|---|---|
| 0 | `L960`: YOLO11l, 960 px | largest affordable model (~1.3x YOLO11m compute) | ~2.4 |
| 0, then | `M960b`: YOLO11m, 960 px, other seed, stronger scale augmentation (0.7) | same strength as 10, different errors | ~1.8 |
| 1 | `M1280`: YOLO11m, 1280 px | small objects (AP_small is the weakest part) | ~3.3 |

All: 04's best config (box 12, dfl 2), 12 epochs, cosine schedule, as notebook 10 (no cross-validation: the official val
decides). Each new model gets the six-pass inference of 07-10 on val and test.

**Fusion** (notebook 10's rule, now 6 members: `04`, `08`, `10` and the new ones), staged search on val (7 clips, kept small):
score power `p` in {1, 2, 3} -> each member x2 or left out, in turn (a second round only if the first helped) -> matching IoU.
Check first: notebook 10's fusion is reproduced exactly.

**Go / no-go:** submit only if the best fusion beats notebook 10's fusion on val by at least **+0.005**.

**Protections**
- every training and every prediction runs in its own process, pinned to one GPU; the main process never uses the GPU;
- GPU memory check per model before its training; the batch is halved automatically if memory is too tight;
- **time guard:** a model starts only if it can finish (with the speed measured so far) within `BUDGET_H`;
  with one GPU only, the same code runs one lane and the guard drops the model that does not fit;
- if one lane fails, the other continues and the fusion uses the members that finished (warning printed);
- final weights and passes are saved as soon as each model is done; re-running with this notebook's output attached
  reuses them (then no GPU is needed for the fusion).

**Inputs:** competition; outputs of **01** (`vd_prep`), **07** (`submission_07_A2_A3_wbf.csv`), **08** (`passes_08_val.pkl.gz`,
`submission_08_A2_A3_wbf.csv`), **09** (`passes_final_04_val.pkl.gz`), **10** (`passes_10_val.pkl.gz`, `submission_10_yolo11m.csv`,
`results_10_val.csv`). Accelerator **GPU T4 x2**, **Internet ON**. Per-process logs: `/kaggle/working/logs/`.

In [1]:
%pip install -q ultralytics

In [2]:
import os

import os, io, json, time, math, random, shutil, contextlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

SEED = 0
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
import subprocess
try:
    _smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:
    _smi = []
GPUS = [(n.strip(), float(m) / 1024) for n, m in (l.split(",") for l in _smi)]      # (name, GB) per GPU
print("GPUs:", GPUS or "NONE -> choose the accelerator 'GPU T4 x2' in the session options")

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_all(pattern, depth=5):
    # search attached inputs at increasing nesting depth (Kaggle nests inputs differently per source)
    for d in range(depth):
        hit = sorted(INPUT.glob("/".join(["*"] * d + [pattern])))
        if hit:
            return hit
    return []

def find(pattern):
    hit = find_all(pattern)
    if not hit:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}: check the attached inputs")
    return hit[0]

PREP = find("vd_prep/config.json").parent            # output of notebook 01
CFG = json.load(open(PREP / "config.json"))
TEST_CSV = find("test_images.csv"); ROOT = TEST_CSV.parent
VAL_DIR = find("val/annotations_coco.json").parent
FOLD = CFG["folds"]                                   # train clip -> fold id (GroupKFold by scene, from notebook 01)
METRICS = ["mAP", "AP50", "AP_s", "AP_m", "AP_l", "ms_per_img"]

import gc, gzip, pickle, sys, threading, ast
from multiprocessing import Pool
from ultralytics import YOLO, settings
settings.update({"wandb": False})

# ---- new members (lane = GPU); all get 04's best config, 12 epochs, cosine schedule (as notebook 10)
NEW = {"L960":  dict(build="yolo11l.pt", imgsz=960, batch=4),
       "M1280": dict(build="yolo11m.pt", imgsz=1280, batch=4),
       "M960b": dict(build="yolo11m.pt", imgsz=960, batch=4, seed=1, scale=0.7)}
EST_H = {"L960": 2.4, "M1280": 3.3, "M960b": 1.8}         # rough train + passes hours on one T4 (from notebook 10's run)
LANES_2GPU = {0: ["L960", "M960b"], 1: ["M1280"]}
LANES_1GPU = {0: ["L960", "M1280", "M960b"]}               # one GPU: the time guard drops what does not fit
BUDGET_H = 5.5                                             # lanes must end within this many hours
EPOCHS = 12
DEV_ARG = 0                                                # device inside a worker (each worker sees only its GPU)
N_GPU = len(GPUS)
GPU_GB = min([g for _, g in GPUS] or [16.0])

# ---- inference and fusion (as in notebooks 07-10)
UP_SCALE, TILE_ZOOM, TILE_OVERLAP = 1.33, 1.5, 0.2
NMS_IOU, WBF_IOU, MAX_DET = 0.6, 0.55, 500
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]
MERGE_WORKERS = max(1, (os.cpu_count() or 2))             # CPU processes for WBF / fusion (after the lanes)
GO_MARGIN = 0.005                                          # required val gain over notebook 10's fusion
VAL_STEP = 5

# ---- inputs: fail here, not after hours
need = {"submission_07_A2_A3_wbf.csv": "07", "passes_08_val.pkl.gz": "08", "submission_08_A2_A3_wbf.csv": "08",
        "passes_final_04_val.pkl.gz": "09", "passes_10_val.pkl.gz": "10", "submission_10_yolo11m.csv": "10",
        "results_10_val.csv": "10"}
missing = [f"{f} (notebook {src})" for f, src in need.items() if not find_all(f)]
assert not missing, "attach the outputs that contain: " + ", ".join(missing)
done_new = [k for k in NEW if all(find_all(f"passes_11_{k}_{s}.pkl.gz") for s in ("val", "test"))]
assert N_GPU or len(done_new) == len(NEW), "no GPU: choose the accelerator 'GPU T4 x2' (or attach this notebook's finished output)"
if N_GPU == 1:
    print("WARNING: only one GPU -> one lane; the time guard will drop the model(s) that do not fit (choose 'GPU T4 x2')")

GPUs: [('Tesla T4', 15.0), ('Tesla T4', 15.0)]


### Data (same as notebooks 04-10)

In [3]:
def load_gt(path):
    # COCO ground truth; ignored regions (iscrowd=1) are copied to all 8 classes so they ignore detections of any class
    d = json.load(open(path))
    crowd = [a for a in d["annotations"] if a.get("iscrowd")]
    d["annotations"] = [a for a in d["annotations"] if not a.get("iscrowd")]
    nid = max([a["id"] for a in d["annotations"]], default=0) + 1
    for a in crowd:
        for c in range(1, 9):
            d["annotations"].append({**a, "id": nid, "category_id": c, "iscrowd": 1, "area": a["bbox"][2] * a["bbox"][3]})
            nid += 1
    gt = COCO()
    gt.dataset = d
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

def coco_eval(gt, dets, img_ids):
    # standard COCO bbox metrics, up to 500 detections per image (as in the leaderboard)
    if not dets:
        return dict(mAP=0.0, AP50=0.0, AP_s=0.0, AP_m=0.0, AP_l=0.0)
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(gt, gt.loadRes(dets), "bbox")
        E.params.imgIds = list(img_ids)
        E.params.maxDets = [1, 10, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats                                  # s[1], s[3:6] already use maxDets[-1] = 500
    prec = E.eval["precision"][:, :, :, 0, -1]   # [IoU, recall, class] for all sizes at 500 dets (pycocotools' s[0] hardcodes 100)
    prec = prec[prec > -1]
    return dict(mAP=float(prec.mean()) if prec.size else -1.0, AP50=s[1], AP_s=s[3], AP_m=s[4], AP_l=s[5])

def write_submission(dets, path):
    # one row per test image: "class score x y w h ..." (0-based class, pixels, top-left corner, max 500 boxes)
    test = pd.read_csv(TEST_CSV)
    by = {}
    for d in dets:
        by.setdefault(d["image_id"], []).append(d)
    rows = []
    for iid in test.image_id:
        top = sorted(by.get(iid, []), key=lambda d: -d["score"])[:500]
        rows.append(" ".join(f"{d['category_id'] - 1} {d['score']:.4f} " + " ".join(f"{v:.1f}" for v in d["bbox"]) for d in top))
    sub = pd.DataFrame({"image_id": test.image_id, "PredictionString": rows})
    sub.to_csv(path, index=False)
    sample = find_all("sample_submission.csv")      # not shipped with this competition's data: ids come from test_images.csv
    if sample:
        assert list(sub.image_id) == list(pd.read_csv(sample[0]).image_id), "image ids differ from sample_submission.csv"
    print(f"saved {path}: {len(sub)} rows, {sum(map(bool, rows))} with detections, {sum(len(r.split()) // 6 for r in rows)} boxes")

In [4]:
# training frames exported by notebook 01 (ignored regions already grayed out)
TJ = json.load(open(PREP / "train_coco.json"))
IMGS = {i["id"]: i for i in TJ["images"]}
GT_TRAIN = load_gt(PREP / "train_coco.json")

def fold_ids(f):
    # (train ids, held-out ids) for fold f; f=None -> all frames for training
    return ([i for i, m in IMGS.items() if FOLD[m["video"]] != f],
            [i for i, m in IMGS.items() if FOLD[m["video"]] == f])

def copy_parallel(src, dst, workers=16):
    # copy a folder file by file with many threads (on network storage the per-file latency dominates)
    from concurrent.futures import ThreadPoolExecutor
    files = [p for p in Path(src).rglob("*") if p.is_file()]
    def cp(p):
        t = Path(dst) / p.relative_to(src)
        t.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(p, t)
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(cp, files))

# /kaggle/input is network storage (~5 MB/s, ~50 ms per file): copy the training images once to local disk
IMG_ROOT = PREP / "images" / "train"
if str(PREP).startswith("/kaggle/input"):
    local = Path("/tmp/vd_prep/images/train")
    if not (local.parent / ".copied").exists():
        t0 = time.time()
        copy_parallel(IMG_ROOT, local)
        (local.parent / ".copied").touch()
        print(f"copied {sum(1 for _ in local.iterdir())} training images to local disk in {time.time() - t0:.0f} s")
    IMG_ROOT = local

def train_path(i):
    return IMG_ROOT / IMGS[i]["file_name"]

# official val: raw frames at original resolution, every VAL_STEP-th frame of each clip (for speed)
VAL_GT = load_gt(VAL_DIR / "annotations_coco.json")
_idx = {(p.parent.name, p.name): p for p in (VAL_DIR / "images").rglob("*.jpg")}
_v = pd.DataFrame(VAL_GT.dataset["images"])
_v = _v[(_v.groupby("video").frame_id.rank(method="first").astype(int) - 1) % VAL_STEP == 0]
VAL_ITEMS = [(int(r.id), _idx[(r.video, Path(r.file_name).name)]) for r in _v.itertuples()]
VAL_IDS = [i for i, _ in VAL_ITEMS]
TEST_ITEMS = [(r.image_id, ROOT / r.file_name) for r in pd.read_csv(TEST_CSV).itertuples()]
print(f"{len(IMGS)} train frames | {len(VAL_ITEMS)} val frames | {len(TEST_ITEMS)} test frames")

copied 4868 training images to local disk in 13 s
4868 train frames | 571 val frames | 3811 test frames


In [5]:
import sys, subprocess

DEVICE_ID = globals().get("DEVICE_ID", 0)
# Ultralytics finds labels by swapping images -> labels in the path, and writes a .cache next to them
IMG_DIR = IMG_ROOT                                          # local copy of the training images (data cell)
LBL_DIR = IMG_DIR.parent.parent / "labels" / "train"
if not LBL_DIR.exists():
    copy_parallel(PREP / "labels" / "train", LBL_DIR)
JOB_DIR = Path("/tmp/yolo_jobs") if str(WORK).startswith("/kaggle") else WORK / "yolo_jobs"   # image lists, yamls, job files
JOB_DIR.mkdir(parents=True, exist_ok=True)

def write_data(name, tr_ids, va_ids):
    # Ultralytics data yaml with explicit image lists, so folds are just different lists
    for part, ids in (("train", tr_ids), ("val", va_ids)):
        (JOB_DIR / f"{name}_{part}.txt").write_text("\n".join(str(IMG_DIR / IMGS[i]["file_name"]) for i in ids))
    q = lambda x: json.dumps(str(x))                        # quoted path, safe for Windows backslashes
    p = JOB_DIR / f"{name}.yaml"
    p.write_text(f"path: {q(JOB_DIR)}\ntrain: {q(JOB_DIR / (name + '_train.txt'))}\nval: {q(JOB_DIR / (name + '_val.txt'))}\nnames: {json.dumps(CFG['classes'])}\n")
    return str(p)

# Every training and every prediction runs in its own short-lived Python process. When it exits, all of its
# GPU memory is released, so nothing piles up between runs (inside one process it did, and the GPU ran out).
WORKER = JOB_DIR / "ultra_worker.py"
WORKER.write_text("""
import json, sys
job = json.load(open(sys.argv[1]))
import torch
from ultralytics import YOLO, RTDETR, settings
settings.update({"wandb": False})
Cls = {"YOLO": YOLO, "RTDETR": RTDETR}[job["cls"]]
out = {}
if job["kind"] == "train":
    model = Cls(job["build"])
    if job.get("load"):
        model = model.load(job["load"])                     # built from a yaml: transfer the pretrained weights
    model.train(**job["kwargs"])
else:
    from ultralytics.utils import DEFAULT_CFG_DICT
    kw = dict(job["kwargs"])
    if job.get("fp16"):                                     # newer versions renamed half=True to quantize=16
        kw.update({"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True})
    model = Cls(job["weights"])
    items = json.load(open(job["items"]))
    dets, ms, CHUNK = [], [], 16
    # small chunks: Ultralytics 8.4 loads a whole list of paths as ONE batch (ignores `batch`) -> out of memory
    for k in range(0, len(items), CHUNK):
        chunk = items[k:k + CHUNK]
        for (iid, _), r in zip(chunk, model.predict([p for _, p in chunk], stream=True, **kw)):
            b = r.boxes
            xywh = b.xyxy.cpu().numpy()
            xywh[:, 2:] -= xywh[:, :2]
            for box, s, c in zip(xywh.tolist(), b.conf.tolist(), b.cls.tolist()):
                dets.append([iid, int(c) + 1, *[round(v, 2) for v in box], round(s, 5)])
            ms.append(r.speed["inference"] + r.speed["postprocess"])
    out["dets"], out["ms"] = dets, sum(ms) / max(len(ms), 1)
if torch.cuda.is_available():
    out["peak_gb"] = torch.cuda.max_memory_reserved() / 1e9
json.dump(out, open(job["out"], "w"))
""")

def run_worker(job, tag):
    # runs one job in a fresh process, prints its log here, returns its result
    job = {**job, "out": str(JOB_DIR / f"{tag}.out.json")}
    jp = JOB_DIR / f"{tag}.job.json"
    jp.write_text(json.dumps(job))
    proc = subprocess.Popen([sys.executable, str(WORKER), str(jp)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{tag}: the worker process failed, see its log above")
    return json.loads(Path(job["out"]).read_text())

def fit(spec, name, tr_ids, va_ids, **hp):
    kw = {"data": write_data(name, tr_ids, va_ids), "project": str(WORK / "runs"), "name": name, **{**COMMON, **hp}}
    res = run_worker({**spec, "kind": "train", "kwargs": kw}, name)
    (WORK / "curves").mkdir(exist_ok=True)
    shutil.copy(WORK / "runs" / name / "results.csv", WORK / "curves" / f"{TAG}_{name}.csv")     # per-epoch val curve
    # last epoch, not "best": best is picked on the held-out fold
    return WORK / "runs" / name / "weights" / "last.pt", res.get("peak_gb", float("nan"))

def detect(weights, items, imgsz):
    # returns COCO-style detections in original pixel coordinates and model ms per image (inference + postprocess)
    tag = f"predict_{Path(weights).parent.parent.name}_{len(items)}"
    ip = JOB_DIR / f"{tag}.items.json"
    ip.write_text(json.dumps([[i, str(p)] for i, p in items]))
    kw = dict(imgsz=imgsz, conf=0.001, iou=0.6, max_det=500, batch=4, verbose=False, device=DEVICE_ID)
    res = run_worker({"cls": MODEL_CLS.__name__, "kind": "predict", "weights": str(weights), "items": str(ip),
                      "kwargs": kw, "fp16": True}, tag)
    dets = [{"image_id": d[0], "category_id": d[1], "bbox": d[2:6], "score": d[6]} for d in res["dets"]]
    return dets, res["ms"]

def eval_fold(weights, ids, imgsz):
    dets, ms = detect(weights, [(i, IMG_DIR / IMGS[i]["file_name"]) for i in ids], imgsz)
    return {**coco_eval(GT_TRAIN, dets, ids), "ms_per_img": ms}

### The lanes: training + six-pass inference of each new model, one lane per GPU
Each job is a separate Python process that sees only its lane's GPU (`CUDA_VISIBLE_DEVICES`); its output goes to
`logs/<job>.log`. A status line is printed here every 10 minutes. This cell is the long one (~4-5 h).

In [6]:
# six-pass inference worker: the code of notebooks 07-10, run in its own process on one GPU
PASSES_WORKER = JOB_DIR / "passes_worker.py"
PASSES_WORKER.write_text("""
import json, sys, gzip, pickle, math, os
import numpy as np, cv2, torch
from torchvision.ops import batched_nms
from ultralytics import YOLO, settings
from ultralytics.utils import DEFAULT_CFG_DICT
settings.update({"wandb": False})
job = json.load(open(sys.argv[1]))
globals().update(job["consts"])
DEVICE_ID = job["device"]
PREC = ({"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True}) if torch.cuda.is_available() else {}
""" + """
def load_model(path):
    # model + the image size it was trained at
    ck = torch.load(path, map_location="cpu", weights_only=False)
    imgsz = int(np.max(ck.get("train_args", {}).get("imgsz", 960)))
    del ck
    return YOLO(str(path)), imgsz

def run(model, images, imgsz):
    # one model call on a few images (Ultralytics 8.4 puts a whole list into one batch); one (N, 6) array per image
    res = model.predict(images, imgsz=imgsz, conf=0.001, iou=0.6, max_det=MAX_DET, verbose=False, device=DEVICE_ID, **PREC)
    return [np.c_[r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()].astype(np.float32) for r in res]

def unflip(b, width):
    # boxes predicted on a horizontally flipped image -> original orientation
    b = b.copy(); b[:, [0, 2]] = width - b[:, [2, 0]]
    return b

def nms(b, iou):
    # class-wise NMS, best first, at most MAX_DET boxes (main process only)
    if not len(b):
        return b
    t = torch.from_numpy(b)
    keep = batched_nms(t[:, :4], t[:, 4], t[:, 5].long(), iou).numpy()
    return b[keep[:MAX_DET]]

def tile_grid(W, H):
    # overlapping crops of side max(W, H) / TILE_ZOOM (clipped to the frame), at least TILE_OVERLAP overlap
    T = int(max(W, H) / TILE_ZOOM)
    tw, th = min(T, W), min(T, H)
    def starts(size, t):
        n = max(1, math.ceil((size - t) / (t * (1 - TILE_OVERLAP))) + 1) if size > t else 1
        return np.linspace(0, size - t, n).round().astype(int)
    return [(x, y, x + tw, y + th) for y in starts(H, th) for x in starts(W, tw)]

ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def predict_views(model, imgsz, img):
    # all six passes of one frame, boxes in original pixel coordinates
    H, W = img.shape[:2]
    up = int(round(imgsz * UP_SCALE / 32) * 32)
    flip = np.ascontiguousarray(img[:, ::-1])
    out = {"full": run(model, [img], imgsz)[0], "full_flip": unflip(run(model, [flip], imgsz)[0], W),
           "up": run(model, [img], up)[0], "up_flip": unflip(run(model, [flip], up)[0], W)}
    grid = tile_grid(W, H)
    for name in ("tiles", "tiles_flip"):
        crops = [img[y0:y1, x0:x1] for x0, y0, x1, y1 in grid]
        if name == "tiles_flip":
            crops = [np.ascontiguousarray(c[:, ::-1]) for c in crops]
        parts = []
        for (x0, y0, x1, y1), b in zip(grid, run(model, crops, imgsz)):
            if name == "tiles_flip":
                b = unflip(b, x1 - x0)
            # drop boxes cut by an inner crop edge: the neighbouring crop (>= 20 % overlap) or the full pass has the whole object
            e = 2
            cut = (((b[:, 0] <= e) & (x0 > 0)) | ((b[:, 2] >= x1 - x0 - e) & (x1 < W)) |
                   ((b[:, 1] <= e) & (y0 > 0)) | ((b[:, 3] >= y1 - y0 - e) & (y1 < H)))
            b = b[~cut].copy(); b[:, [0, 2]] += x0; b[:, [1, 3]] += y0
            parts.append(b)
        out[name] = nms(np.concatenate(parts), NMS_IOU)       # remove duplicates from overlapping crops
    return out

def passes(model_path, tag, items):
    # prediction passes of one model on a set of frames; reused from this or an attached output, else computed and saved
    fname = f"passes_{tag}.pkl.gz"
    hit = [p for p in [WORK / fname] if p.exists()] + find_all(fname)
    if hit:
        with gzip.open(hit[0], "rb") as f:
            cache = pickle.load(f)
        if set(cache) == {i for i, _ in items}:
            print(f"reusing {hit[0]}")
            return cache
        print(f"{hit[0]} covers other frames, recomputing")
    model, imgsz = load_model(model_path)
    print(f"{tag}: {Path(model_path).name} trained at {imgsz} px, {len(items)} frames")
    cache = {iid: predict_views(model, imgsz, cv2.imread(str(p))) for iid, p in tqdm(items, desc=tag)}
    del model; gc.collect(); torch.cuda.empty_cache()
    with gzip.open(WORK / fname, "wb") as f:
        pickle.dump(cache, f, protocol=4)
    return cache

""" + """
model, imgsz = load_model(job["weights"])
for part in job["parts"]:
    items = json.load(open(part["items"]))
    cache = {}
    for k, (iid, p) in enumerate(items):
        cache[iid] = predict_views(model, imgsz, cv2.imread(p))
        if k % 500 == 0:
            print(part["out"], k, "/", len(items), flush=True)
    tmp = part["out"] + ".tmp"
    with gzip.open(tmp, "wb") as f:
        pickle.dump(cache, f, protocol=4)
    os.replace(tmp, part["out"])                            # complete file or none
out = {"peak_gb": torch.cuda.max_memory_reserved() / 1e9} if torch.cuda.is_available() else {}
json.dump(out, open(job["out"], "w"))
""")
LOG_DIR = WORK / "logs"; LOG_DIR.mkdir(exist_ok=True)
CONSTS = dict(MAX_DET=MAX_DET, UP_SCALE=UP_SCALE, TILE_ZOOM=TILE_ZOOM, TILE_OVERLAP=TILE_OVERLAP, NMS_IOU=NMS_IOU)
PRINT_LOCK = threading.Lock()
def say(*a):
    with PRINT_LOCK:
        print(time.strftime("%H:%M"), *a, flush=True)

def launch(script, job, tag, gpu):
    # one job in a fresh process that sees only GPU `gpu`; output -> logs/<tag>.log; returns the job's result
    job = {**job, "out": str(JOB_DIR / f"{tag}.out.json")}
    jp = JOB_DIR / f"{tag}.job.json"
    jp.write_text(json.dumps(job))
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)} if N_GPU else dict(os.environ)
    with open(LOG_DIR / f"{tag}.log", "w") as log:
        rc = subprocess.run([sys.executable, str(script), str(jp)], stdout=log, stderr=subprocess.STDOUT, env=env).returncode
    if rc != 0:
        tail = (LOG_DIR / f"{tag}.log").read_text(errors="replace").splitlines()[-40:]
        raise RuntimeError(f"{tag} failed (exit code {rc}); end of logs/{tag}.log:\n" + "\n".join(tail))
    return json.loads(Path(job["out"]).read_text())

def mine_or_attached(fname):
    return [p for p in [WORK / fname] if p.exists()] + find_all(fname)

def epochs_done(run):
    p = WORK / "runs" / run / "results.csv"
    return max(0, len(p.read_text().strip().splitlines()) - 1) if p.exists() else 0

TR_ALL, LOG_VAL = fold_ids(None)[0], fold_ids(0)[1][:200]  # all training frames; 200 frames only feed the training log
n_obj = {}
for a in TJ["annotations"]:
    if not a.get("iscrowd"):
        n_obj[a["image_id"]] = n_obj.get(a["image_id"], 0) + 1
DENSE = sorted(IMGS, key=lambda i: -n_obj.get(i, 0))[:64]   # most crowded frames, for the memory check
ITEM_FILES = {}
for split, items in (("val", VAL_ITEMS), ("test", TEST_ITEMS)):
    ITEM_FILES[split] = JOB_DIR / f"items_{split}.json"
    ITEM_FILES[split].write_text(json.dumps([[i, str(p)] for i, p in items]))

def write_lane_data(name, tr_ids, va_ids, gpu):
    # like write_data, but each lane reads the images through its own folder (symlinks), so Ultralytics writes its
    # label cache (labels/train.cache) per lane: two trainings starting together never read a half-written cache
    root = JOB_DIR / f"lane{gpu}"
    for kind, src in (("images", IMG_DIR), ("labels", LBL_DIR)):
        d = root / kind / "train"
        if not d.exists():
            d.parent.mkdir(parents=True, exist_ok=True)
            d.symlink_to(src, target_is_directory=True)
    img = root / "images" / "train"
    for part, ids in (("train", tr_ids), ("val", va_ids)):
        (JOB_DIR / f"{name}_{part}.txt").write_text("\n".join(str(img / IMGS[i]["file_name"]) for i in ids))
    q = lambda x: json.dumps(str(x))
    p = JOB_DIR / f"{name}.yaml"
    p.write_text(f"path: {q(JOB_DIR)}\ntrain: {q(JOB_DIR / (name + '_train.txt'))}\nval: {q(JOB_DIR / (name + '_val.txt'))}\nnames: {json.dumps(CFG['classes'])}\n")
    return str(p)

def train_kwargs(name, run, data, **over):
    c = NEW[name]
    workers = max(2, (os.cpu_count() or 4) // max(1, len(LANES)))
    kw = dict(data=data, project=str(WORK / "runs"), name=run, epochs=EPOCHS, batch=c["batch"], imgsz=c["imgsz"],
              workers=workers, device=DEV_ARG, seed=c.get("seed", SEED), warmup_epochs=2, close_mosaic=3,
              box=12.0, dfl=2.0, cos_lr=True, plots=False, exist_ok=True)
    if "scale" in c:
        kw["scale"] = c["scale"]
    return {**kw, **over}

def build_member(name, gpu):
    # final weights -> final_11_<name>.pt, then its val / test passes -> passes_11_<name>_{val,test}.pkl.gz
    final = WORK / f"final_11_{name}.pt"
    hit = mine_or_attached(final.name)
    run = f"final_11_{name}"
    if hit:
        if hit[0] != final:
            shutil.copy(hit[0], final)
        say(name, "final weights reused")
    elif epochs_done(run) >= EPOCHS:
        shutil.copy(WORK / "runs" / run / "weights" / "last.pt", final)
        say(name, "finished training run reused")
    else:
        bs = NEW[name]["batch"]
        while True:                                         # GPU memory check: one epoch on the 64 most crowded frames
            kw = train_kwargs(name, f"mem_{name}", write_lane_data(f"mem_{name}", DENSE, DENSE[:16], gpu),
                              epochs=1, warmup_epochs=0, close_mosaic=0, batch=bs)
            res = launch(WORKER, {"cls": "YOLO", "build": NEW[name]["build"], "kind": "train", "kwargs": kw}, f"mem_{name}_b{bs}", gpu)
            shutil.rmtree(WORK / "runs" / f"mem_{name}", ignore_errors=True)
            peak = res.get("peak_gb", 0.0)
            say(name, f"memory check: batch {bs} -> peak {peak:.1f} of {GPU_GB:.1f} GB")
            if peak < 0.85 * GPU_GB:
                break
            if bs <= 2:
                raise RuntimeError(f"{name}: too close to the GPU memory limit even at batch 2")
            bs //= 2
        NEW[name]["batch"] = bs
        t0 = time.time()
        say(name, f"training on {len(TR_ALL)} frames, batch {bs}")
        kw = train_kwargs(name, run, write_lane_data(run, TR_ALL, LOG_VAL, gpu), batch=bs)
        launch(WORKER, {"cls": "YOLO", "build": NEW[name]["build"], "kind": "train", "kwargs": kw}, run, gpu)
        shutil.copy(WORK / "runs" / run / "weights" / "last.pt", final)   # last epoch, as in 04-10
        (WORK / "curves").mkdir(exist_ok=True)
        shutil.copy(WORK / "runs" / run / "results.csv", WORK / "curves" / f"yolo11_{run}.csv")
        TIMES[name]["train_min"] = (time.time() - t0) / 60
        say(name, f"trained in {TIMES[name]['train_min']:.0f} min")
    parts = []
    for split in ("val", "test"):
        fname = f"passes_11_{name}_{split}.pkl.gz"
        hit = mine_or_attached(fname)
        if hit:
            if hit[0] != WORK / fname:
                shutil.copy(hit[0], WORK / fname)
        else:
            parts.append({"items": str(ITEM_FILES[split]), "out": str(WORK / fname)})
    if parts:
        t0 = time.time()
        launch(PASSES_WORKER, {"weights": str(final), "device": DEV_ARG, "consts": CONSTS, "parts": parts}, f"passes_{name}", gpu)
        TIMES[name]["passes_min"] = (time.time() - t0) / 60
        say(name, f"six passes on {', '.join(Path(p['out']).name for p in parts)} in {TIMES[name]['passes_min']:.0f} min")

def speed_ratio():
    # measured / estimated hours of the models built here so far (1 if none yet)
    r = [(t["train_min"] + t.get("passes_min", 0)) / 60 / EST_H[k] for k, t in TIMES.items() if "train_min" in t]
    return max(1.0, float(np.mean(r))) if r else 1.0

def lane(gpu, names):
    for name in names:
        ready = mine_or_attached(f"final_11_{name}.pt") and all(mine_or_attached(f"passes_11_{name}_{s}.pkl.gz") for s in ("val", "test"))
        el = (time.time() - T_START) / 3600
        need_h = 0 if ready else EST_H[name] * speed_ratio()
        if not ready and el + need_h > BUDGET_H:
            say(f"GPU {gpu}: {name} skipped by the time guard ({el:.1f} h elapsed + ~{need_h:.1f} h > {BUDGET_H} h)")
            STATE[name] = "skipped (time)"
            continue
        STATE[name] = "running"
        try:
            build_member(name, gpu)
            STATE[name] = "done"
        except Exception as e:                              # this lane goes on with its next model, the other lane is unaffected
            STATE[name] = "FAILED"
            say(f"GPU {gpu}: {name} FAILED:\n{e}")

LANES = LANES_2GPU if N_GPU >= 2 else LANES_1GPU
for b in sorted({c["build"] for c in NEW.values()}):        # download the pretrained weights once (lanes start at the same time)
    YOLO(b)
TIMES = {k: {} for k in NEW}
STATE = {k: "waiting" for k in NEW}
T_START = time.time()
threads = []
for gpu, names in LANES.items():
    t = threading.Thread(target=lane, args=(gpu, names), daemon=True)
    t.start(); threads.append(t)
    time.sleep(90 if N_GPU else 1)                          # stagger the starts (first-run downloads and caches)
last = 0
while any(t.is_alive() for t in threads):
    time.sleep(30)
    if time.time() - last > 600:
        last = time.time()
        say(f"{(time.time() - T_START) / 3600:.1f} h |", " | ".join(
            f"{k}: {STATE[k]}" + (f" epoch {epochs_done('final_11_' + k)}/{EPOCHS}" if STATE[k] == "running" else "") for k in NEW))
NEW_OK = [k for k in NEW if all((WORK / f"passes_11_{k}_{s}.pkl.gz").exists() for s in ("val", "test"))]
say(f"lanes finished in {(time.time() - T_START) / 3600:.2f} h | new members ready: {NEW_OK} | states: {STATE}")
if len(NEW_OK) < len(NEW):
    print("WARNING: not all new models finished; the fusion uses the ones that did (re-run with this output attached to add the rest)")
assert NEW_OK, "no new model finished: see the messages above and logs/"

15:39 L960 memory check: batch 4 -> peak 7.7 of 15.0 GB
15:39 L960 training on 4868 frames, batch 4
15:40 M1280 memory check: batch 4 -> peak 11.3 of 15.0 GB
15:40 M1280 training on 4868 frames, batch 4
15:41 0.1 h | L960: running epoch 0/12 | M1280: running epoch 0/12 | M960b: waiting
15:51 0.2 h | L960: running epoch 1/12 | M1280: running epoch 1/12 | M960b: waiting
16:01 0.4 h | L960: running epoch 3/12 | M1280: running epoch 2/12 | M960b: waiting
16:11 0.6 h | L960: running epoch 4/12 | M1280: running epoch 3/12 | M960b: waiting
16:21 0.7 h | L960: running epoch 6/12 | M1280: running epoch 4/12 | M960b: waiting
16:31 0.9 h | L960: running epoch 7/12 | M1280: running epoch 5/12 | M960b: waiting
16:41 1.1 h | L960: running epoch 9/12 | M1280: running epoch 6/12 | M960b: waiting
16:51 1.2 h | L960: running epoch 10/12 | M1280: running epoch 7/12 | M960b: waiting
16:59 L960 trained in 81 min
17:01 1.4 h | L960: running epoch 12/12 | M1280: running epoch 8/12 | M960b: waiting
17:11 1.6 

### Members: merged six-pass outputs on val and test

In [7]:
def wbf(views, iou_thr=WBF_IOU):
    # views: list of (N, 6) arrays, one per pass; returns fused (M, 6), best first, at most MAX_DET
    n_views = len(views)
    allb = np.concatenate([v for v in views if len(v)] or [np.zeros((0, 6), np.float32)])
    out = []
    for c in np.unique(allb[:, 5]):
        b = allb[allb[:, 5] == c]
        b = b[np.argsort(-b[:, 4])]
        acc = np.zeros((len(b), 4)); fb = np.zeros((len(b), 4)); w = np.zeros(len(b)); cnt = np.zeros(len(b)); m = 0
        for x1, y1, x2, y2, s, _ in b:
            if m:
                f = fb[:m]
                ix = np.clip(np.minimum(f[:, 2], x2) - np.maximum(f[:, 0], x1), 0, None)
                iy = np.clip(np.minimum(f[:, 3], y2) - np.maximum(f[:, 1], y1), 0, None)
                inter = ix * iy
                iou = inter / ((f[:, 2] - f[:, 0]) * (f[:, 3] - f[:, 1]) + (x2 - x1) * (y2 - y1) - inter + 1e-9)
                j = int(np.argmax(iou))
                if iou[j] > iou_thr:
                    acc[j] += s * np.array([x1, y1, x2, y2]); w[j] += s; cnt[j] += 1
                    fb[j] = acc[j] / w[j]
                    continue
            acc[m] = s * np.array([x1, y1, x2, y2]); w[m] = s; cnt[m] = 1; fb[m] = (x1, y1, x2, y2); m += 1
        score = w[:m] / cnt[:m] * np.minimum(cnt[:m], n_views) / n_views
        out.append(np.c_[fb[:m], score, np.full(m, c)])
    if not out:
        return np.zeros((0, 6), np.float32)
    res = np.concatenate(out).astype(np.float32)
    return res[np.argsort(-res[:, 4])][:MAX_DET]

def merge(views, how):
    if how is None:                                         # single pass: as predicted
        return views[0][:MAX_DET]
    if how == "nms":
        return nms(np.concatenate(views), NMS_IOU)
    return wbf(views)

STRATEGIES = {                                              # name: (views, merge)
    "S0_single_pass":   (["full"], None),
    "A1_upscaled_pass": (["up"], None),
    "A2_tta_nms":       (["full", "full_flip", "up", "up_flip"], "nms"),
    "A2_tta_wbf":       (["full", "full_flip", "up", "up_flip"], "wbf"),
    "A3_tiles_nms":     (["full", "tiles"], "nms"),
    "A3_tiles_wbf":     (["full", "tiles"], "wbf"),
    "A2+A3_wbf":        (["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"], "wbf"),
}
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def merge_task(args):
    # one image, one strategy (runs in a worker process)
    iid, views, how = args
    t0 = time.perf_counter()
    b = merge(views, how)
    return iid, b, 1000 * (time.perf_counter() - t0)

def merge_all(cache, views, how):
    # merge the cached passes of every frame; returns detections (COCO style) and merge ms per frame
    tasks = [(iid, [cache[iid][v] for v in views], how) for iid in cache]
    # worker processes only for WBF (pure NumPy). NMS uses PyTorch, which can hang inside forked processes
    # once its thread pool was used in the main process; NMS takes only seconds here anyway.
    if MERGE_WORKERS and how == "wbf":
        with Pool(MERGE_WORKERS) as p:
            res = p.map(merge_task, tasks, chunksize=8)
    else:
        res = [merge_task(t) for t in tasks]
    dets = [{"image_id": iid, "category_id": int(r[5]) + 1, "bbox": [float(r[0]), float(r[1]), float(r[2] - r[0]), float(r[3] - r[1])],
             "score": float(r[4])} for iid, b, _ in res for r in b]
    return dets, float(np.mean([t for _, _, t in res]))


def pmap(f, tasks, chunk):
    # NumPy-only work in MERGE_WORKERS processes (serial with 0)
    if not MERGE_WORKERS:
        return [f(t) for t in tasks]
    with Pool(MERGE_WORKERS) as p:
        return p.map(f, tasks, chunksize=chunk)

def merged(cache, ids):
    # six-pass WBF of each image (strategy A2+A3_wbf), rounded like the submission CSVs (score 4, box 1 decimal)
    tasks = [(iid, [cache[iid][v] for v in ALL_VIEWS], "wbf") for iid in ids]
    res = pmap(merge_task, tasks, 8)
    return {iid: csv_round(b) for iid, b, _ in res}

def csv_round(b):
    # same precision as a submission row: the val numbers then match what the test CSVs contain
    b = b.copy().astype(np.float64)
    wh = np.round(b[:, 2:4] - b[:, 0:2], 1); b[:, 0:2] = np.round(b[:, 0:2], 1); b[:, 2:4] = b[:, 0:2] + wh
    b[:, 4] = np.round(b[:, 4], 4)
    return b.astype(np.float32)

def to_dets(by):
    return [{"image_id": iid, "category_id": int(r[5]) + 1, "bbox": [float(r[0]), float(r[1]), float(r[2] - r[0]), float(r[3] - r[1])],
             "score": float(r[4])} for iid, b in by.items() for r in b]


def read_submission(path):
    # submission CSV -> {image_id: (N, 6) [x1, y1, x2, y2, score, class]}
    out = {}
    for iid, s in pd.read_csv(path).fillna("").itertuples(index=False):
        v = np.array(s.split(), dtype=np.float64).reshape(-1, 6)       # class score x y w h
        out[iid] = np.c_[v[:, 2], v[:, 3], v[:, 2] + v[:, 4], v[:, 3] + v[:, 5], v[:, 1], v[:, 0]].astype(np.float32)
    return out


def load_passes(fname):
    with gzip.open(mine_or_attached(fname)[0], "rb") as f:
        return pickle.load(f)

TEST_IDS = [i for i, _ in TEST_ITEMS]
VAL, TEST = {}, {}
for k, f in (("04", "passes_final_04_val.pkl.gz"), ("08", "passes_08_val.pkl.gz"), ("10", "passes_10_val.pkl.gz")):
    c = load_passes(f); VAL[k] = merged(c, VAL_IDS); del c
TEST["04"] = read_submission(find("submission_07_A2_A3_wbf.csv"))
TEST["08"] = read_submission(find("submission_08_A2_A3_wbf.csv"))
TEST["10"] = read_submission(find("submission_10_yolo11m.csv"))
for k in NEW_OK:
    c = load_passes(f"passes_11_{k}_val.pkl.gz"); VAL[k] = merged(c, VAL_IDS); del c
    c = load_passes(f"passes_11_{k}_test.pkl.gz"); TEST[k] = merged(c, TEST_IDS); del c
    gc.collect()
    write_submission(to_dets(TEST[k]), WORK / f"submission_11_{k}.csv")
for k in TEST:
    assert set(TEST[k]) == set(TEST_IDS), f"member {k}: test output covers other images than test_images.csv"
print("members:", list(VAL))

saved /kaggle/working/submission_11_L960.csv: 3811 rows, 3811 with detections, 1617709 boxes
saved /kaggle/working/submission_11_M1280.csv: 3811 rows, 3811 with detections, 1619391 boxes
saved /kaggle/working/submission_11_M960b.csv: 3811 rows, 3811 with detections, 1694490 boxes
members: ['04', '08', '10', 'L960', 'M1280', 'M960b']


### Fusion across all members (notebook 10's rule)

In [8]:
def fuse(parts, w, iou_thr, p):
    # parts: one (N, 6) array [x1, y1, x2, y2, score, class] per model; w: model weights (members with weight 0 are left out)
    keep = [k for k in range(len(parts)) if w[k] > 0]
    parts, w = [parts[k] for k in keep], np.asarray([w[k] for k in keep], float)
    w = w / w.sum(); M = len(parts)
    allb = np.concatenate([np.c_[a, np.full(len(a), k)] for k, a in enumerate(parts)]) if sum(map(len, parts)) else np.zeros((0, 7))
    out = []
    for c in np.unique(allb[:, 5]):
        x = allb[allb[:, 5] == c]
        x = x[np.argsort(-x[:, 4] * w[x[:, 6].astype(int)], kind="stable")]   # best weighted score first
        n = len(x)
        acc = np.zeros((n, 4)); fb = np.zeros((n, 4)); sw = np.zeros(n); best = np.zeros((n, M)); m = 0
        for x1, y1, x2, y2, s, _, k in x:
            k = int(k); ws = s * w[k]
            if m:
                f = fb[:m]
                ix = np.clip(np.minimum(f[:, 2], x2) - np.maximum(f[:, 0], x1), 0, None)
                iy = np.clip(np.minimum(f[:, 3], y2) - np.maximum(f[:, 1], y1), 0, None)
                inter = ix * iy
                iou = inter / ((f[:, 2] - f[:, 0]) * (f[:, 3] - f[:, 1]) + (x2 - x1) * (y2 - y1) - inter + 1e-9)
                iou[best[:m, k] > 0] = 0                               # at most one box per model in a fused box
                j = int(np.argmax(iou))
                if iou[j] > iou_thr:
                    acc[j] += ws * np.array([x1, y1, x2, y2]); sw[j] += ws; fb[j] = acc[j] / sw[j]
                    best[j, k] = s
                    continue
            acc[m] = ws * np.array([x1, y1, x2, y2]); sw[m] = ws; fb[m] = (x1, y1, x2, y2); best[m] = 0; best[m, k] = s; m += 1
        found = best[:m] > 0
        share = (found * w).sum(1)                                     # weight share of the models that found the box
        mean = (best[:m] * w).sum(1) / share                           # weighted mean score of those models
        out.append(np.c_[fb[:m], mean * share ** p, np.full(m, c)])
    if not out:
        return np.zeros((0, 6), np.float32)
    r = np.concatenate(out).astype(np.float32)
    return r[np.argsort(-r[:, 4])][:MAX_DET]

def fuse_task(args):
    iid, parts, w, iou_thr, p = args
    return iid, fuse(parts, w, iou_thr, p)

def fuse_all(outs, ids, w, iou_thr, p):
    # outs: list of {image_id: (N, 6)}, one per member
    empty = np.zeros((0, 6), np.float32)
    return dict(pmap(fuse_task, [(i, [o.get(i, empty) for o in outs], w, iou_thr, p) for i in ids], 16))


NAMES = list(VAL)
rows = []
def score(label, by, **extra):
    m = coco_eval(VAL_GT, to_dets(by), VAL_IDS)
    rows.append({"variant": label, **m, **extra})
    print(f"{label:60s} mAP {m['mAP']:.4f} | AP_s {m['AP_s']:.4f}", flush=True)
    return m["mAP"]

solo = {k: score(f"{k} alone", VAL[k]) for k in NAMES}
r10 = pd.read_csv(find("results_10_val.csv")).set_index("variant")
r10 = r10[r10.index.str.startswith("fusion ")]
b10 = r10.mAP.idxmax(); REF_10 = float(r10.loc[b10, "mAP"])
W10 = [float(x) for x in ast.literal_eval(r10.loc[b10, "w"])]
N10 = [t.split("=")[0] for t in b10.split()[1:] if "=" in t and not t.startswith(("iou=", "p="))]
assert N10 == ["04", "08", "10"], f"notebook 10's best fusion uses members {N10}"
fus10 = score(f"10 fusion ({b10[7:]})", fuse_all([VAL[k] for k in N10], VAL_IDS, W10, float(r10.loc[b10, "iou"]), float(r10.loc[b10, "p"])))
assert abs(fus10 - REF_10) < 0.002, f"10's fusion gives {fus10:.4f} here, notebook 10 reported {REF_10:.4f}"

04 alone                                                     mAP 0.2046 | AP_s 0.1645
08 alone                                                     mAP 0.2066 | AP_s 0.1288
10 alone                                                     mAP 0.2258 | AP_s 0.1450
L960 alone                                                   mAP 0.2531 | AP_s 0.1428
M1280 alone                                                  mAP 0.2465 | AP_s 0.1486
M960b alone                                                  mAP 0.2190 | AP_s 0.1294
10 fusion (04=1 08=1 10=2 iou=0.6 p=3)                       mAP 0.2469 | AP_s 0.1812


In [9]:
tried = {}
def trial(w, iou_thr, p):
    key = (tuple(w), iou_thr, p)
    if key not in tried:
        label = "fusion " + " ".join(f"{k}={v:g}" for k, v in zip(NAMES, w)) + f" iou={iou_thr} p={p}"
        tried[key] = score(label, fuse_all([VAL[k] for k in NAMES], VAL_IDS, list(w), iou_thr, p), w=list(w), iou=iou_thr, p=p)
    return tried[key]

def coordinate_round(W, iou_thr, p):
    # each member x2 or left out, in turn, starting from W; returns the best weights found
    best = W
    for i in range(len(NAMES)):
        for f in (2.0, 0.0):
            w = tuple(W[j] * f if j == i else W[j] for j in range(len(NAMES)))
            if sum(w) > 0 and trial(w, iou_thr, p) > trial(best, iou_thr, p):
                best = w
    return best

t0 = time.time()
EQUAL = tuple([1.0] * len(NAMES))
P = max((1, 2, 3), key=lambda p: trial(EQUAL, 0.55, p))                 # stage 1: penalty for boxes found by few models
W = coordinate_round(EQUAL, 0.55, P)                                    # stage 2: member weights
if W != EQUAL and trial(W, 0.55, P) - trial(EQUAL, 0.55, P) > 0.001:
    W = coordinate_round(W, 0.55, P)                                    # second round only if the first helped
IOU = max((0.55, 0.5, 0.6), key=lambda t: trial(W, t, P))              # stage 3: matching IoU
best_map = tried[(W, IOU, P)]
res_val = pd.DataFrame(rows).set_index("variant")
res_val.to_csv(WORK / "results_11_val.csv")
display(res_val[["mAP", "AP50", "AP_s", "AP_m", "AP_l"]].round(4))
print(f"search: {len(tried)} fusions in {(time.time() - t0) / 60:.0f} min | best:", dict(zip(NAMES, W)), f"iou={IOU} p={P} -> val mAP {best_map:.4f}")

fusion 04=1 08=1 10=1 L960=1 M1280=1 M960b=1 iou=0.55 p=1    mAP 0.2701 | AP_s 0.1572
fusion 04=1 08=1 10=1 L960=1 M1280=1 M960b=1 iou=0.55 p=2    mAP 0.2694 | AP_s 0.1651
fusion 04=1 08=1 10=1 L960=1 M1280=1 M960b=1 iou=0.55 p=3    mAP 0.2667 | AP_s 0.1680
fusion 04=2 08=1 10=1 L960=1 M1280=1 M960b=1 iou=0.55 p=1    mAP 0.2662 | AP_s 0.1616
fusion 04=0 08=1 10=1 L960=1 M1280=1 M960b=1 iou=0.55 p=1    mAP 0.2714 | AP_s 0.1562
fusion 04=1 08=2 10=1 L960=1 M1280=1 M960b=1 iou=0.55 p=1    mAP 0.2670 | AP_s 0.1566
fusion 04=1 08=0 10=1 L960=1 M1280=1 M960b=1 iou=0.55 p=1    mAP 0.2706 | AP_s 0.1610
fusion 04=1 08=1 10=2 L960=1 M1280=1 M960b=1 iou=0.55 p=1    mAP 0.2674 | AP_s 0.1590
fusion 04=1 08=1 10=0 L960=1 M1280=1 M960b=1 iou=0.55 p=1    mAP 0.2656 | AP_s 0.1555
fusion 04=1 08=1 10=1 L960=2 M1280=1 M960b=1 iou=0.55 p=1    mAP 0.2738 | AP_s 0.1580
fusion 04=1 08=1 10=1 L960=0 M1280=1 M960b=1 iou=0.55 p=1    mAP 0.2579 | AP_s 0.1569
fusion 04=1 08=1 10=1 L960=1 M1280=2 M960b=1 iou=0.55 

,mAP,AP50,AP_s,AP_m,AP_l
variant,,,,,
04 alone,0.2046,0.4058,0.1645,0.2059,0.3278
08 alone,0.2066,0.4159,0.1288,0.2127,0.3080
10 alone,0.2258,0.4353,0.1450,0.2165,0.3667
L960 alone,0.2531,0.4679,0.1428,0.2261,0.3916
M1280 alone,0.2465,0.4869,0.1486,0.2473,0.3741
M960b alone,0.2190,0.4327,0.1294,0.2219,0.3221
10 fusion (04=1 08=1 10=2 iou=0.6 p=3),0.2469,0.4776,0.1812,0.2365,0.3892
fusion 04=1 08=1 10=1 L960=1 M1280=1 M960b=1 iou=0.55 p=1,0.2701,0.5125,0.1572,0.2576,0.4101
fusion 04=1 08=1 10=1 L960=1 M1280=1 M960b=1 iou=0.55 p=2,0.2694,0.5130,0.1651,0.2584,0.4108


search: 28 fusions in 70 min | best: {'04': 0.0, '08': 1.0, '10': 1.0, 'L960': 2.0, 'M1280': 1.0, 'M960b': 1.0} iou=0.5 p=1 -> val mAP 0.2757


**Observations:**
- Model size is the strongest lever so far: YOLO11l (960 px) alone reaches 0.253 on val, better than notebook 10's whole 3-model fusion (0.247). YOLO11m at 1280 px reaches 0.247 (best AP50 0.487 and AP_medium 0.247).
- The second YOLO11m with another seed and stronger scale augmentation (0.7) is worse than notebook 10's YOLO11m (0.219 vs 0.226), so the stronger augmentation did not help.
- The search keeps p = 1 (with more members, boxes found by few models need less penalty) and up-weights YOLO11l (x2). It leaves out final_04 (0.2757 vs 0.2738 with it).
- Equal weights over all six give 0.270; the searched weights give +0.006 on top. The search tried 28 fusions on 7 val clips, so part of this is fitted to val (checked in notebook 12).

### Test submission and verdict

In [10]:
fused = fuse_all([TEST[k] for k in NAMES], TEST_IDS, list(W), IOU, P)
write_submission(to_dets(fused), WORK / "submission_11_ensemble.csv")

gain = best_map - REF_10
go = gain >= GO_MARGIN
print(f"\nval mAP: 10 fusion {REF_10:.4f} | best new single model {max(solo[k] for k in NEW_OK):.4f} | "
      f"best fusion {best_map:.4f} ({gain:+.4f} vs 10)")
print("VERDICT:", "GO -> submit submission_11_ensemble.csv" if go else
      f"NO-GO -> gain below +{GO_MARGIN} on val: keep notebook 10's submission (0.2869)")
summary = pd.DataFrame(rows).assign(split="val")
summary["train_min"] = summary.variant.map(lambda v: TIMES.get(v.split()[0], {}).get("train_min", np.nan))
summary["passes_min"] = summary.variant.map(lambda v: TIMES.get(v.split()[0], {}).get("passes_min", np.nan))
summary.to_csv(WORK / "results_11_summary.csv", index=False)

saved /kaggle/working/submission_11_ensemble.csv: 3811 rows, 3811 with detections, 1875886 boxes
val mAP: 10 fusion 0.2469 | best new single model 0.2531 | best fusion 0.2757 (+0.0288 vs 10)
VERDICT: GO -> submit submission_11_ensemble.csv


**Observations:**
- GO (+0.029 on val over notebook 10). Leaderboard 0.2938 vs 0.2869 (+0.007): only about a quarter of the val gain transferred, the weakest transfer so far.
- Training both lanes in parallel on two T4s worked as planned: 3.1 h for three models (81, 119 and 63 min training + 17-28 min inference each). The fusion search took another 70 min on the CPU of the GPU session.